# 多人地下城与勇士

本笔记本展示了 `DialogueAgent` 和 `DialogueSimulator` 类如何轻松地将[双人地下城与勇士示例](https://python.langchain.com/en/latest/use_cases/agent_simulations/two_player_dnd.html)扩展到多人游戏。

模拟双人游戏与多人游戏的主要区别在于修改每个代理发言的时间表。

为此，我们增强了 `DialogueSimulator`，使其能够接受一个自定义函数，该函数决定了哪个代理发言的时间表。在下面的示例中，每个角色轮流发言，故事叙述者穿插在每个玩家之间。

## 导入 LangChain 相关模块

In [1]:
from typing import Callable, List

from langchain.schema import (
    HumanMessage,
    SystemMessage,
)
from langchain_openai import ChatOpenAI

## `DialogueAgent` 类
`DialogueAgent` 类是对 `ChatOpenAI` 模型的简单封装，它通过简单地将消息串联成字符串来存储来自 `dialogue_agent` 的消息历史。

它公开了两个方法：
- `send()`: 将聊天模型应用于消息历史并返回消息字符串
- `receive(name, message)`: 将 `name` 所说的 `message` 添加到消息历史中

In [2]:
class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")

## `DialogueSimulator` 类
`DialogueSimulator` 类接收一个代理列表。在每一步，它会执行以下操作：
1. 选择下一位发言者
2. 调用下一位发言者发送消息
3. 将消息广播给所有其他代理
4. 更新步数计数器。
下一位发言者的选择可以实现为任何函数，但在本例中，我们只是循环遍历代理。

In [3]:
class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message

## 定义角色和任务

In [4]:
character_names = ["Harry Potter", "Ron Weasley", "Hermione Granger", "Argus Filch"]
storyteller_name = "Dungeon Master"
quest = "Find all of Lord Voldemort's seven horcruxes."
word_limit = 50  # word limit for task brainstorming

## 让 LLM 为游戏描述添加细节

In [5]:
game_description = f"""Here is the topic for a Dungeons & Dragons game: {quest}.
        The characters are: {*character_names,}.
        The story is narrated by the storyteller, {storyteller_name}."""

player_descriptor_system_message = SystemMessage(
    content="You can add detail to the description of a Dungeons & Dragons player."
)


def generate_character_description(character_name):
    character_specifier_prompt = [
        player_descriptor_system_message,
        HumanMessage(
            content=f"""{game_description}
            Please reply with a creative description of the character, {character_name}, in {word_limit} words or less. 
            Speak directly to {character_name}.
            Do not add anything else."""
        ),
    ]
    character_description = ChatOpenAI(temperature=1.0)(
        character_specifier_prompt
    ).content
    return character_description


def generate_character_system_message(character_name, character_description):
    return SystemMessage(
        content=(
            f"""{game_description}
    Your name is {character_name}. 
    Your character description is as follows: {character_description}.
    You will propose actions you plan to take and {storyteller_name} will explain what happens when you take those actions.
    Speak in the first person from the perspective of {character_name}.
    For describing your own body movements, wrap your description in '*'.
    Do not change roles!
    Do not speak from the perspective of anyone else.
    Remember you are {character_name}.
    Stop speaking the moment you finish speaking from your perspective.
    Never forget to keep your response to {word_limit} words!
    Do not add anything else.
    """
        )
    )


character_descriptions = [
    generate_character_description(character_name) for character_name in character_names
]
character_system_messages = [
    generate_character_system_message(character_name, character_description)
    for character_name, character_description in zip(
        character_names, character_descriptions
    )
]

storyteller_specifier_prompt = [
    player_descriptor_system_message,
    HumanMessage(
        content=f"""{game_description}
        Please reply with a creative description of the storyteller, {storyteller_name}, in {word_limit} words or less. 
        Speak directly to {storyteller_name}.
        Do not add anything else."""
    ),
]
storyteller_description = ChatOpenAI(temperature=1.0)(
    storyteller_specifier_prompt
).content

storyteller_system_message = SystemMessage(
    content=(
        f"""{game_description}
You are the storyteller, {storyteller_name}. 
Your description is as follows: {storyteller_description}.
The other players will propose actions to take and you will explain what happens when they take those actions.
Speak in the first person from the perspective of {storyteller_name}.
Do not change roles!
Do not speak from the perspective of anyone else.
Remember you are the storyteller, {storyteller_name}.
Stop speaking the moment you finish speaking from your perspective.
Never forget to keep your response to {word_limit} words!
Do not add anything else.
"""
    )
)

In [6]:
print("Storyteller Description:")
print(storyteller_description)
for character_name, character_description in zip(
    character_names, character_descriptions
):
    print(f"{character_name} Description:")
    print(character_description)

Storyteller Description:
Dungeon Master, your power over this adventure is unparalleled. With your whimsical mind and impeccable storytelling, you guide us through the dangers of Hogwarts and beyond. We eagerly await your every twist, your every turn, in the hunt for Voldemort's cursed horcruxes.
Harry Potter Description:
"Welcome, Harry Potter. You are the young wizard with a lightning-shaped scar on your forehead. You possess brave and heroic qualities that will be essential on this perilous quest. Your destiny is not of your own choosing, but you must rise to the occasion and destroy the evil horcruxes. The wizarding world is counting on you."
Ron Weasley Description:
Ron Weasley, you are Harry's loyal friend and a talented wizard. You have a good heart but can be quick to anger. Keep your emotions in check as you journey to find the horcruxes. Your bravery will be tested, stay strong and focused.
Hermione Granger Description:
Hermione Granger, you are a brilliant and resourceful wi

## 使用 LLM 创建详尽的任务描述

This guide will walk you through using a Large Language Model (LLM) to create an elaborate quest description for your game.

本指南将引导您使用大型语言模型 (LLM) 为您的游戏创建详尽的任务描述。

### Prerequisites

在开始之前，您需要：

*   **An LLM:** This could be a locally hosted model (like Llama 2, Mistral) or an API-based model (like OpenAI's GPT-3.5/GPT-4, Anthropic's Claude).
    *   **一个 LLM：** 这可以是一个本地托管的模型（如 Llama 2、Mistral）或一个基于 API 的模型（如 OpenAI 的 GPT-3.5/GPT-4、Anthropic 的 Claude）。
*   **A basic understanding of prompt engineering:** Knowing how to structure your requests to the LLM will yield better results.
    *   **基本的提示工程知识：** 了解如何构建发送给 LLM 的请求将产生更好的结果。

### Steps

步骤：

1.  **Define the Core Quest:**
    *   **确定核心任务：**
    *   What is the basic goal of the quest? (e.g., "Retrieve a stolen artifact," "Defeat a menacing beast," "Deliver a message to a distant town.")
        *   任务的基本目标是什么？（例如，“取回被盗的文物”、“击败一只凶猛的野兽”、“向遥远的城镇传递信息”。）
    *   Who gives the quest? (e.g., a worried villager, a powerful king, a mysterious hermit.)
        *   谁发布了任务？（例如，一位忧心忡忡的村民、一位强大的国王、一位神秘的隐士。）
    *   What is the reward? (e.g., gold, a magic item, reputation, information.)
        *   奖励是什么？（例如，金币、魔法物品、声望、信息。）

2.  **Craft Your Initial Prompt:**
    *   **构建您的初始提示：**
    *   Start with a clear instruction.
        *   从清晰的指令开始。
    *   Provide the core quest details.
        *   提供核心任务的详细信息。
    *   Specify the desired output format and tone.
        *   指定期望的输出格式和语气。

    **Example Prompt:**

    **示例提示：**

    ```
    Act as a game master. Create an elaborate quest description for a fantasy RPG.

    Quest Giver: Elara, a worried village elder.
    Objective: Retrieve the Sunstone, a sacred artifact stolen from the village shrine by goblins.
    Location: The Sunstone was last seen heading towards the Whispering Caves, north of the village.
    Reward: 100 gold coins and a Potion of Greater Healing.
    Tone: Mysterious, slightly urgent, and adventurous.
    Include:
    - A compelling title.
    - A brief background story about the Sunstone and its importance.
    - Details about the quest giver and their plea.
    - Hints about the dangers in the Whispering Caves.
    - A clear statement of the objective and reward.
    ```

    ```
    扮演一位游戏主持人。为一款奇幻角色扮演游戏创建一个详尽的任务描述。

    任务发布者：Elara，一位忧心忡忡的村庄长老。
    目标：取回太阳石，这是被地精从村庄圣坛偷走的神圣文物。
    地点：太阳石最后被看到是朝着村庄北面的低语洞穴方向去的。
    奖励：100 枚金币和一瓶高等治疗药水。
    语气：神秘、略带紧迫感、充满冒险气息。
    包含：
    - 一个引人入胜的标题。
    - 关于太阳石及其重要性的简短背景故事。
    - 关于任务发布者及其恳求的详细信息。
    - 关于低语洞穴中危险的提示。
    - 对目标和奖励的清晰陈述。
    ```

3.  **Iterate and Refine:**
    *   **迭代和优化：**
    *   Run the prompt through your LLM.
        *   通过您的 LLM 运行提示。
    *   Review the output. Does it meet your expectations?
        *   审查输出。它是否符合您的期望？
    *   If not, adjust your prompt. You might need to:
        *   如果不符合，请调整您的提示。您可能需要：
        *   Be more specific about certain elements (e.g., "Describe the goblins as cunning and territorial").
            *   对某些元素更加具体（例如，“将地精描述为狡猾且有领地意识的”）。
        *   Ask for additional details (e.g., "Add a rumour about a specific trap in the caves").
            *   要求更多细节（例如，“添加一个关于洞穴中特定陷阱的传闻”）。
        *   Change the requested tone or style.
            *   更改请求的语气或风格。

    **Example Refinement Prompt:**

    **示例优化提示：**

    ```
    That's a good start. Now, please expand on the description of Elara. Make her sound frail but determined. Also, add a detail about a specific type of creature that might be encountered in the Whispering Caves, besides goblins.
    ```

    ```
    这是一个不错的开始。现在，请扩展 Elara 的描述。让她听起来虚弱但意志坚定。另外，除了地精之外，请添加一个关于在低语洞穴中可能遇到的特定类型生物的细节。
    ```

4.  **Add Interactivity (Optional):**
    *   **添加互动性（可选）：**
    *   You can prompt the LLM to include elements that encourage player choice or investigation.
        *   您可以提示 LLM 包含鼓励玩家选择或调查的元素。
    *   Examples:
        *   示例：
        *   "Include a moral dilemma related to retrieving the artifact."
            *   “包含一个与取回文物相关的道德困境。”
        *   "Suggest potential alternative approaches to completing the quest."
            *   “提出完成任务的潜在替代方法。”
        *   "Add a hidden clue that might lead to a secret area."
            *   “添加一个可能通往秘密区域的隐藏线索。”

### Example LLM-Generated Quest Description

### 示例 LLM 生成的任务描述

Here's an example of what the LLM might generate after a few iterations:

以下是 LLM 在几次迭代后可能生成的示例：

```markdown
# The Sunstone's Shadow

**Quest Giver:** Elara, Village Elder

**Background:**
For generations, the Sunstone has been the heart of Oakhaven, a beacon of warmth and prosperity. This fist-sized gem, pulsating with gentle golden light, rests within the village shrine, warding off the encroaching gloom of the Whispering Woods. But three nights ago, under the cloak of a new moon, shadowy figures – goblins from the northern hills – infiltrated the shrine and stole the Sunstone. Without its light, a chilling dampness creeps into our homes, and the crops begin to wither.

**The Plea:**
Elara, her face etched with worry and her hands trembling, beckons you closer. "Brave adventurer," she rasps, her voice thin as parchment, "our village is fading. The Sunstone is our lifeblood. The goblins took it towards the Whispering Caves, a place of ill repute and ancient dangers. We are simple folk, unable to venture there ourselves. Please, retrieve our sacred stone before Oakhaven is lost to the darkness forever. We offer all we can spare."

**The Objective:**
Venture into the treacherous Whispering Caves and recover the stolen Sunstone.

**The Dangers:**
The Whispering Caves are a labyrinth of damp, echoing tunnels. Goblins, known for their cunning ambushes and crude traps, are sure to be present. Rumours also speak of **Gloomfang Spiders**, large arachnids that spin webs of shadow and possess a venom that induces terrifying hallucinations. Be wary of unstable rock formations and the unnerving whispers that give the caves their name – some say they are the voices of lost souls.

**The Reward:**
Upon your successful return with the Sunstone, Elara will reward you with **100 gold coins** and a **Potion of Greater Healing**, a potent brew that can mend even grievous wounds. Your bravery will also be sung by the hearths of Oakhaven for seasons to come.
```

```markdown
# 太阳石的阴影

**任务发布者：** Elara，村庄长老

**背景：**
数代以来，太阳石一直是橡树湾的心脏，是温暖和繁荣的灯塔。这颗拳头大小的宝石，脉动着柔和的金色光芒，安放在村庄的圣坛中，抵御着低语森林日益逼近的阴霾。但三天前，在新月之夜的掩护下，黑影——来自北方山丘的地精——潜入了圣坛，偷走了太阳石。没有了它的光芒，一股令人不寒而栗的潮湿侵入了我们的家园，庄稼也开始枯萎。

**恳求：**
Elara，她的脸上刻满了忧虑，双手颤抖着，向你招手。“勇敢的冒险者，”她嘶哑地说，声音像羊皮纸一样细弱，“我们的村庄正在衰败。太阳石是我们的生命线。地精们把它带往了低语洞穴，一个声名狼藉且充满古老危险的地方。我们是普通人，无法亲自前往。拜托了，在橡树湾永远迷失于黑暗之前，请找回我们神圣的石头。我们愿献出我们所能节省的一切。”

**目标：**
深入险恶的低语洞穴，找回被盗的太阳石。

**危险：**
低语洞穴是一个潮湿、回荡的隧道迷宫。地精以其狡猾的伏击和粗糙的陷阱而闻名，肯定会出现在那里。谣言还提到了**暗影獠牙蜘蛛**，一种巨大的蜘蛛，它们吐出的蛛网充满阴影，并且拥有能引起可怕幻觉的毒液。要小心不稳定的岩石构造和赋予洞穴其名称的令人不安的低语——有人说那是迷失灵魂的声音。

**奖励：**
在你成功带回太阳石后，Elara 将奖励你 **100 枚金币** 和一瓶 **高等治疗药水**，这是一种强大的药剂，可以治愈严重的伤势。你的勇气也将被橡树湾的人们在炉火旁传唱数季。

In [7]:
quest_specifier_prompt = [
    SystemMessage(content="You can make a task more specific."),
    HumanMessage(
        content=f"""{game_description}
        
        You are the storyteller, {storyteller_name}.
        Please make the quest more specific. Be creative and imaginative.
        Please reply with the specified quest in {word_limit} words or less. 
        Speak directly to the characters: {*character_names,}.
        Do not add anything else."""
    ),
]
specified_quest = ChatOpenAI(temperature=1.0)(quest_specifier_prompt).content

print(f"Original quest:\n{quest}\n")
print(f"Detailed quest:\n{specified_quest}\n")

Original quest:
Find all of Lord Voldemort's seven horcruxes.

Detailed quest:
Harry Potter and his companions must journey to the Forbidden Forest, find the hidden entrance to Voldemort's secret lair, and retrieve the horcrux guarded by the deadly Acromantula, Aragog. Remember, time is of the essence as Voldemort's power grows stronger every day. Good luck.



## 主循环

The main loop is the core of the application. It's where all the events are processed.

主循环是应用程序的核心。它负责处理所有事件。

```jsx
function App() {
  const [count, setCount] = useState(0);

  // The main loop is implicitly handled by React's rendering cycle.
  // When the state changes, React re-renders the component.
  // This re-rendering process is analogous to the main loop
  // in traditional event-driven applications.

  return (
    <div>
      <p>You clicked {count} times</p>
      <button onClick={() => setCount(count + 1)}>
        Click me
      </button>
    </div>
  );
}
```

{/*
The main loop is the central part of an application that continuously
waits for and processes events. In many traditional programming
paradigms, this is explicitly written as a `while` loop.

In a framework like React, the concept of a "main loop" is abstracted
away. React manages its own rendering cycle, which is driven by state
changes and user interactions. When you call `setCount`, React schedules
a re-render. This re-render process is where your component's JSX is
evaluated and the UI is updated. This is the React equivalent of a
main loop.
*/}

{/*
In a traditional event-driven application, the main loop might look
something like this:

```javascript
while (true) {
  // Wait for an event
  const event = waitForEvent();

  // Process the event
  processEvent(event);
}
```

React's internal mechanisms handle this pattern for you. When you
interact with a component (e.g., clicking a button), an event is
triggered. React's event system captures this event and updates the
component's state. This state update then triggers a re-render,
effectively restarting the "loop" for that component.
*/}

In [8]:
characters = []
for character_name, character_system_message in zip(
    character_names, character_system_messages
):
    characters.append(
        DialogueAgent(
            name=character_name,
            system_message=character_system_message,
            model=ChatOpenAI(temperature=0.2),
        )
    )
storyteller = DialogueAgent(
    name=storyteller_name,
    system_message=storyteller_system_message,
    model=ChatOpenAI(temperature=0.2),
)

In [9]:
def select_next_speaker(step: int, agents: List[DialogueAgent]) -> int:
    """
    If the step is even, then select the storyteller
    Otherwise, select the other characters in a round-robin fashion.

    For example, with three characters with indices: 1 2 3
    The storyteller is index 0.
    Then the selected index will be as follows:

    step: 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16

    idx:  0  1  0  2  0  3  0  1  0  2  0  3  0  1  0  2  0
    """
    if step % 2 == 0:
        idx = 0
    else:
        idx = (step // 2) % (len(agents) - 1) + 1
    return idx

In [10]:
max_iters = 20
n = 0

simulator = DialogueSimulator(
    agents=[storyteller] + characters, selection_function=select_next_speaker
)
simulator.reset()
simulator.inject(storyteller_name, specified_quest)
print(f"({storyteller_name}): {specified_quest}")
print("\n")

while n < max_iters:
    name, message = simulator.step()
    print(f"({name}): {message}")
    print("\n")
    n += 1

(Dungeon Master): Harry Potter and his companions must journey to the Forbidden Forest, find the hidden entrance to Voldemort's secret lair, and retrieve the horcrux guarded by the deadly Acromantula, Aragog. Remember, time is of the essence as Voldemort's power grows stronger every day. Good luck.


(Harry Potter): I suggest we sneak into the Forbidden Forest under the cover of darkness. Ron, Hermione, and I can use our wands to create a Disillusionment Charm to make us invisible. Filch, you can keep watch for any signs of danger. Let's move quickly and quietly.


(Dungeon Master): As you make your way through the Forbidden Forest, you hear the eerie sounds of nocturnal creatures. Suddenly, you come across a clearing where Aragog and his spider minions are waiting for you. Ron, Hermione, and Harry, you must use your wands to cast spells to fend off the spiders while Filch keeps watch. Be careful not to get bitten!


(Ron Weasley): I'll cast a spell to create a fiery blast to scare off